In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
ee_key_host_path =r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json' # Path in the host machine
ee_key_container_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

volumes = {
    ee_key_host_path: {'bind': ee_key_container_path, 'mode': 'ro'},  # 'ro' means read-only
}
from dask_cluster_manager import DaskClusterManager

dcm = DaskClusterManager()
dcm.create_test_cluster(volumes=volumes)

Dask Scheduler started with ID 0dd491efd987533f46a6211f28b6e4b7071f2cada5d2e7cbca74a2978ee90612
Dask Worker started with ID 1d91244f61ac3375a72d36275c86bb47c47bba61e1681e25a26662e84992897d
Test cluster created with 1 worker and half the system memory.
Dask dashboard available at http://localhost:8787


In [5]:
from dask.distributed import Client
dask_client = Client("tcp://localhost:8786")

In [3]:
dcm.stop_and_remove_containers()

Dask Worker 937bd7a5de442df521053b6d7169e75ebb1357ac17652379a0127fad3a922d1f stopped and removed.
Dask Worker 47c0a79e1df0f4e3bc62087d23ebe79520331e1a74b886cb6317ba0d6d0264ad stopped and removed.
Dask Worker 6989b346cc876792723d5f1e1d9355f1597774b0dbb500a0ddd18fc48b311d81 stopped and removed.
Dask Worker 3fb92cfe088174d9fd6ed280633afceb27dfe36bedd15972d2960cf86be9d38d stopped and removed.
Dask Worker 938635faba3852fca99533f28ba54b0e9e18de4c58a3d983d639312166ff4610 stopped and removed.
Dask Worker f30519848ffa2a8fa2c393a178c238844e1ad6bdaea11c16337c22c79a7dc62a stopped and removed.
Dask Worker 119437320b085bbcd54262a4440f909f03843c95b92362e7ed39c4c75cae7289 stopped and removed.
Dask Worker 3a0ce3f87cea1b4782ab1158d0221e958dd4a7f1f5c64166a83126bce60cc91d stopped and removed.
Dask Worker b341117f360c70655c11466204d5212eb750c16d48ae14e6b182eac0b95c106b stopped and removed.
Dask Worker 84181026bf0e6dd079d1782e4004762b784f445323cc34ad868ab73c56a23f93 stopped and removed.
Dask Worker fd372cfa

In [ ]:
logs = scheduler.logs()
print(logs.decode("utf-8"))

In [6]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(ee_key_container_path)
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.10:37745': {'status': 'OK'},
 'tcp://172.20.0.11:35777': {'status': 'OK'},
 'tcp://172.20.0.12:38979': {'status': 'OK'},
 'tcp://172.20.0.13:34347': {'status': 'OK'},
 'tcp://172.20.0.14:41273': {'status': 'OK'},
 'tcp://172.20.0.15:41917': {'status': 'OK'},
 'tcp://172.20.0.16:38197': {'status': 'OK'},
 'tcp://172.20.0.17:37541': {'status': 'OK'},
 'tcp://172.20.0.18:45731': {'status': 'OK'},
 'tcp://172.20.0.3:41121': {'status': 'OK'},
 'tcp://172.20.0.4:34763': {'status': 'OK'},
 'tcp://172.20.0.5:45475': {'status': 'OK'},
 'tcp://172.20.0.6:34845': {'status': 'OK'},
 'tcp://172.20.0.7:37423': {'status': 'OK'},
 'tcp://172.20.0.8:40765': {'status': 'OK'},
 'tcp://172.20.0.9:39873': {'status': 'OK'}}

In [7]:
# Earth Engine xarray
import sys
import os
import ee
import json

json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineData

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'map_function': prep_sr_l8
        }

earth_engine = EarthEngineData(parameters)
chunk_size = {'time': 3, 'X': 2048, 'Y': 2048}
earth_engine.chunk_dataset(chunk_size)
#ds = EarthEngineData(parameters).dataset

In [ ]:
chunk_size = {'time': 3, 'X': 9084, 'Y': 10578}
earth_engine.chunk_dataset(chunk_size)

In [ ]:
chunk_size = {'time': 3, 'X': 2048, 'Y': 2048}
earth_engine.chunk_dataset(chunk_size)

In [6]:
print(earth_engine.dataset)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 2048, 2048), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 2048, 2048), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_

In [7]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 1, 'Y': 1}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001C5C981A170>>
Traceback (most recent call last):
  File "c:\Users\Adriano Matos\anaconda3\envs\robust_raster\lib\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
# Load the data using Dask Delayed!
import xarray as xr
import dask

chunk_size  = {
            'time': 3,
            'X': 512,
            'Y': 256
}
# Define a delayed function that opens the dataset
@dask.delayed
def load_dataset_on_worker():
    return xr.open_dataset('/data/saved_on_disk.nc', chunks=chunk_size)

# Trigger the delayed execution, this will happen on the worker
delayed_ds = load_dataset_on_worker()

# Optionally, you can convert the delayed object into a Dask-backed xarray Dataset
ds = delayed_ds.compute()

# Now ds can be used for further computation on the Dask workers

In [ ]:
import xarray as xr
import dask.array as da
import dask
import psutil
import time

def log_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024**2  # Memory usage in MB


def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output


def tune_user_function(chunk, user_func, user_func2, *args, **kwargs):

    result_chunk = user_func2(ds, user_func, *args, kwargs)
    start_time = time.time()
    result_chunk.compute()
    end_time = time.time()
        
    wall_time = end_time - start_time
    memory_usage = log_memory_usage()
    print(f"Wall time: {wall_time:.2f} seconds")
    print(f"Memory usage: {memory_usage:.2f} MB")

    # Apply the processing function to this chunk
    #processed_chunk = user_func(one_chunk)

    return result_chunk

def test_run(ds, user_func, user_func2, *args, **kwargs):
    # Dynamically determine dimension names
    dim_names = list(ds.dims.keys())
            
    # Extract a single chunk to determine the output structure using dynamic dimension names
    one_chunk_slices = {dim: slice(0, ds.chunks[dim][0]) for dim in dim_names}
    one_chunk = ds.isel(**one_chunk_slices)

    test = xr.map_blocks(tune_user_function,
                            one_chunk, 
                            args=(user_func, user_func2) + args, 
                            kwargs=kwargs)

result = test_run(ds, process_chunk_as_pandas, user_function_wrapper)

In [6]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction(earth_engine)

# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
user_defined_func.tune_user_function(earth_engine, process_chunk_as_pandas)

Difference is GREATER than 1%
NEW SLICE: <xarray.Dataset> Size: 472B
Dimensions:  (time: 3, X: 4, Y: 4)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 32B -4.216e+05 -4.215e+05 -4.214e+05 -4.213e+05
  * Y        (Y) float64 32B -5.992e+05 -5.991e+05 -5.99e+05 -5.989e+05
Data variables:
    SR_B4    (time, X, Y) float32 192B dask.array<chunksize=(3, 4, 4), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 192B dask.array<chunksize=(3, 4, 4), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)


KeyboardInterrupt: 

In [4]:
import math
import pandas as pd
import psutil

# Read the CSV file into a DataFrame
df = pd.read_csv('metrics_report.csv')

def get_available_system_memory():
    # Get total available RAM in bytes
    total_ram = psutil.virtual_memory().total

    # Convert from bytes to gigabytes (GB)
    total_ram_gb = total_ram / (1024 ** 3)

    return total_ram_gb

# Get the max workers (RAM limited)
available_system_memory = get_available_system_memory()
ram_safety_threshold = 0.5
max_workers_ram_limited = int(df['wRAM'][0])
memory_per_worker = available_system_memory * ram_safety_threshold / max_workers_ram_limited
print(df['wRAM'][0])
print(available_system_memory)
print(memory_per_worker)
#dcm.stop_and_remove_containers()

ee_key_host_path =r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json' # Path in the host machine
ee_key_container_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

volumes = {
    ee_key_host_path: {'bind': ee_key_container_path, 'mode': 'ro'},  # 'ro' means read-only
}
from dask_cluster_manager import DaskClusterManager
dcm = DaskClusterManager()
dcm.create_cluster(num_workers=max_workers_ram_limited, n_threads=1, memory_limit=f"{memory_per_worker}GB", volumes=volumes)
#dcm.create_cluster(num_workers=int(df['wRAM'][0]), n_threads=1, memory_limit=f"{memory_per_worker}GB", volumes=volumes)

# I need to have a condition that chunks EE Data if I am doing a test or if I am doing a full run
# If test, then chunk to the size of the dataset
# If not, then chunk to 48 MBs?
# We might be limited to 48MBs in Earth Engine...

16
31.767879486083984
0.9927462339401245
Dask Scheduler started with ID 414592c9cfa64892e395fcc59b88f407d99132d6eee6e76d24993cacab1c350e
Dask Worker 1 started with ID 71c419756a129e68d641818eee8e3fc4da9b77970c905daf80858da64c697136
Dask Worker 2 started with ID fa631ad01a1ae60837f353da20d87e13538692f61f6ff0c52fbce5b94d8092b9
Dask Worker 3 started with ID 75f66eddab3a1ce499c5df302acb7566d78c582815eeaaca3e1eff9e6061f8cf
Dask Worker 4 started with ID 0d0a922527cd54ac04ff3952c80c6844bd4567e99166d69cb866f8e3b70ddb1b
Dask Worker 5 started with ID f55966a8e763b04fd8b7ba844e49ca847acd4ff272189f65585523955db7339d
Dask Worker 6 started with ID 2e76db81cdace00cd9914ef62045dd92da31bdf4675f88d2b2db5d75ae5ecb27
Dask Worker 7 started with ID d478f0a363776e199c09e425eb43ac02e139fe2be4304aedb253d572f46c27b9
Dask Worker 8 started with ID 0fbf1e7f022ebb24fc77251acd65910ccfa4fddfaa7b39c03cc987f87ece81e2
Dask Worker 9 started with ID c96f01f4b71a893665bd3fd8ee64f44e3420f020cd622e44593dd228a04da8f2
Dask Wor

In [9]:
dcm.stop_and_remove_containers()

Dask Worker b2abc381f91d0d35872547c62cd614bdf2d8efbb1f0ece514ba994ee5f50a09d stopped and removed.
Dask Worker 94cf99b3ee46839cea4d1922b79f68dbef364b7177a78f789f9e406671f7eb06 stopped and removed.
Dask Worker 64027b86deeae81851b443b52b24353209fae2b3986c5dfa98f28f9a7dd08036 stopped and removed.
Dask Worker d714295e089d2626e4fe51b695a1ebf93bb30ef5f42a291a5a7a7332591c70a4 stopped and removed.
Dask Worker 8801641e585d0ee98a59c19a87f64dc38f2be2b8b6fa989dd0628fae99c8dcc1 stopped and removed.
Dask Worker da704ddd6a9e5d6a77544478307616b66c8d805240854a4ed08d67ca933142ac stopped and removed.
Dask Worker a55d3011d4e5a051048aedd7bbf5c3f79df63649fa1b28e4e0f8ac751bc82cba stopped and removed.
Dask Worker 0bb3e7fd1a9fb8c6642605aa56c76947c7c5b0c95912233305b646251c54033d stopped and removed.
Dask Worker d07f06a4d1134cf1877b7b2b4ae905dfdae5e798aa5bea15c48ffe7deb627419 stopped and removed.
Dask Worker 70c5eda62d21c07bd1568ded81a9b89c80b561f603ae485303cf808f2e27c923 stopped and removed.
Dask Worker fe418864

In [5]:
import xarray as xr
import pandas as pd
import sys, os

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

ds = earth_engine.dataset
# Apply the user-defined function
# If I load the dataset as a dask delayed, I don't need a template!
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

NameError: name 'user_defined_func' is not defined

In [9]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(chunk):
    # Convert xarray chunk to pandas DataFrame
    df = chunk.to_dataframe().reset_index()
    
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    # Convert the pandas DataFrame back to an xarray structure
    df = df.set_index(list(chunk.dims))
    result = df.to_xarray()
    
    return result

# Assuming `your_chunked_xarray` is your chunked xarray object:
result = xr.map_blocks(process_chunk_as_pandas, earth_engine.dataset)
result.compute()

KilledWorker: Attempted to run task ("SR_B5-process_chunk_as_pandas-a7ef4e904b89760b8f39d1bbb9132d55-'xarray-SR_B5-6116b129bf86ae621da794199cda9edc'", 0, 4, 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://172.20.0.12:38469. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
print(result)